# EDA: боты-сборщики vs люди

Цель — понять данные до построения признаков: что сломано в сырых данных, как устроены время и окна,
и какие различия ботов и людей стоит превратить в признаки. Каждый блок заканчивается выводом;
итоговая таблица «наблюдение → признак» — в конце. Все признаки реализованы в `features.py`.

In [ ]:
import numpy as np
import pandas as pd
from features import load_data, clean_events

pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_colwidth', 90)
DATA = 'data'
train, test, events = load_data(DATA)
print('train', train.shape, '| test', test.shape, '| events', events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))

## 1. Структура и гигиена данных

In [ ]:
print('cookie_id уникальны:', train.cookie_id.is_unique, test.cookie_id.is_unique,
      '| пересечение train/test:', len(set(train.cookie_id) & set(test.cookie_id)))
print('окна train:', train.window_start_ts.min().date(), '→', train.window_start_ts.max().date())
print('окна test: ', test.window_start_ts.min().date(), '→', test.window_start_ts.max().date())
print('длина окна:', (train.window_end_ts - train.window_start_ts).unique())
train.groupby(train.window_start_ts.dt.date).target.agg(['size', 'mean']).round(3).T

**Вывод.** Одна строка = одна кука, пересечений train/test нет. Окна суточные. Test (20–26.04) идёт **сразу после**
train (6–19.04), доля ботов по дням стабильна (6–9%) → валидация должна быть **временной** (обучение на прошлых днях,
проверка на следующих), а не случайной.

In [ ]:
print('полные дубли строк:', events.duplicated().sum())
in_order = events.groupby('cookie_id').event_ts.apply(lambda s: s.is_monotonic_increasing).mean()
print('доля кук, у которых события уже упорядочены по времени:', round(in_order, 3))
print('eid ↔ event_name:', events.groupby('event_name').eid.nunique().max() == 1
      and events.groupby('eid').event_name.nunique().max() == 1)
print('\nварианты platform:'); print(events.platform.value_counts(dropna=False))
print('\nдоля пропусков:'); print(events.isna().mean().round(3))

In [ ]:
meta = pd.concat([train.drop(columns='target'), test])
e = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts', 'cookie_created_at']], on='cookie_id')
print('событий до окна:   ', (e.event_ts < e.window_start_ts).sum())
print('событий после окна:', (e.event_ts >= e.window_end_ts).sum())
print('событий в окне:    ', ((e.event_ts >= e.window_start_ts) & (e.event_ts < e.window_end_ts)).sum())

**Вывод — что чинит `clean_events`:**
* файл **не отсортирован** (упорядочено лишь ~3.5% кук) → сортировка по `cookie_id`, `event_ts`; без неё интервалы
  и переходы между событиями были бы мусором;
* ~4.9 тыс. **полных дублей** → удаляем, долю дублей на куку оставляем признаком (`a_dup_share`);
* `platform` записан в разных регистрах, `iphone` = `ios` → нормализация к web / desktop / android / ios;
* `eid` однозначно соответствует `event_name` → `eid` удаляем;
* ~40 тыс. событий лежат **после** окна (до окна — ни одного) → берём только `window_start_ts <= event_ts < window_end_ts`,
  иначе признаки заглядывали бы в будущее;
* пропуски структурные (координаты есть только у части web-событий, `search_page` — только у выдачи) →
  не заполняем, а используем как признак.

## 2. Тайминг: моменты событий, а не их количество

In [ ]:
ev, _ = clean_events(events, meta)
d = ev.merge(train[['cookie_id', 'target']], on='cookie_id')
d['dt'] = d.groupby('cookie_id').event_ts.diff().dt.total_seconds()
d.loc[d.dt > 1800, 'dt'] = np.nan          # паузы > 30 мин — это новая сессия, а не темп
d.groupby('target').dt.describe(percentiles=[.1, .25, .5, .75, .9]).round(1)

In [ ]:
bins = pd.cut(d.dt, [-1, 1, 2, 4, 8, 16, 32, 64, 128, 256, 1800])
pd.crosstab(bins, d.target, normalize='columns').round(3)

**Вывод.** Боты заметно быстрее: медиана интервала ~14 с против ~46 с у людей, пик распределения у ботов — 4–16 с,
у людей — 16–128 с. При этом «ровного таймера» нет (доля повторяющихся интервалов одинакова), поэтому признаки —
квантили и доли коротких интервалов внутри сессии (`t_`). Проверено и отвергнуто: время на странице отдельно по типу
события — после любого события боты уходят одинаково быстро, это дублирует общий тайминг.

## 3. Курсор

In [ ]:
d['has_ptr'] = d.pointer_x.notna()
pd.crosstab([d.platform, d.target], d.has_ptr, normalize='index').round(3)

In [ ]:
p = d[d.has_ptr]
print('равномерное распределение по экрану 1920×1080 имело бы std x ≈ %.0f, std y ≈ %.0f'
      % (1920 / np.sqrt(12), 1080 / np.sqrt(12)))
print(p.groupby('target')[['pointer_x', 'pointer_y']].agg(['mean', 'std', 'min', 'max']).round(1))
step = np.hypot(p.groupby('cookie_id').pointer_x.diff(), p.groupby('cookie_id').pointer_y.diff())
print('\nшаг между соседними точками курсора:')
print(step.groupby(p.target).describe(percentiles=[.25, .5, .75]).round(1))
print('\nдоля точек на границе экрана:',
      ((p.pointer_x.isin([0, 1920])) | (p.pointer_y.isin([0, 1080]))).groupby(p.target).mean().round(3).to_dict())

**Вывод — самый сильный сигнал в данных.**
* На android/iOS координат нет ни у кого; на web/desktop у ботов курсор отсутствует в ~67% событий против ~34% у людей.
* Люди кликают **равномерно** по экрану: std совпадает с равномерным распределением, соседние точки независимы,
  медианный шаг ~745 px.
* Боты — **блуждание**: точки кучкуются (std ~200 px), шаг ~280 px, есть упоры в границы экрана (x = 0 / 1920).

Признаки `p_`: доля пропусков на web, шаг между точками (абсолютный и относительно разброса), автокорреляция x/y,
центр и размах, доля точек на границе.

## 4. Обход контента и выдачи

In [ ]:
c = d.groupby(['cookie_id', 'target']).agg(
    n_events=('event_name', 'size'), n_loc=('item_location', 'nunique'), n_cat=('item_category', 'nunique'),
    n_items=('item_id', 'nunique'), page_max=('search_page', 'max')).reset_index()
c.drop(columns='cookie_id').groupby('target').quantile([.1, .5, .9]).T.round(1)

In [ ]:
pd.crosstab(d.event_name, d.target, normalize='columns').round(3)

**Вывод.** Сборщик обходит шире и глубже: больше городов и категорий, глубже листает выдачу (max page 7 против 4,
90-й перцентиль 20 против 9). Люди чаще листают фото, добавляют в избранное и логинятся. Признаки `c_`, `s_`, `e_`.
Сами объявления (категории, продавцы, `item_id`) у ботов и людей распределены одинаково — важно *как* смотрят,
а не *что*.

## 5. Возраст куки и время суток

In [ ]:
age = (train.window_start_ts - train.cookie_created_at).dt.total_seconds() / 86400
print('возраст куки на начало окна, дней:')
print(age.groupby(train.target).quantile([.1, .5, .9]).unstack().round(1))
print('\nраспределение событий по часам (строки — target):')
pd.crosstab(d.target, d.event_ts.dt.hour, normalize='index').round(3)

**Вывод.** Куки ботов моложе (медиана ~19 дней против ~62) — признак известен к концу окна. Боты заметно активнее
ночью (1–5 ч). Признаки `a_`, `h_`.

## 6. User-Agent

In [ ]:
from features import _parse_ua
ua = d.groupby('user_agent').agg(n_cookies=('cookie_id', 'nunique'), bot_share=('target', 'mean'))
print('уникальных UA:', len(ua), '| UA только у ботов:', (ua.bot_share == 1).sum(),
      '| доля ботов по UA: от %.2f до %.2f' % (ua.bot_share.min(), ua.bot_share.max()))
cookie = d.groupby('cookie_id').agg(ua=('user_agent', 'first'), target=('target', 'first'),
                                    plat=('platform', lambda s: s.mode().iat[0] if s.notna().any() else None))
cookie['family'] = _parse_ua(cookie.ua)['fam']
mob = cookie[cookie.plat.isin(['android', 'ios'])]
print('\nтип клиента на мобильных платформах:')
pd.crosstab(mob.family, mob.target, normalize='columns').round(3)

**Вывод.** Сырая строка UA — плохой признак: ни одна строка не принадлежит только ботам, а как категория она
превращается в идентификатор конкретного сервиса. Поэтому строку **разбираем**: тип клиента (приложение Avito /
браузер), ОС, версия. Содержательная находка: на мобильных ~67% ботов ходят через UA приложения (`Avito/… okhttp`)
против ~38% людей — сбор через API приложения. Плюс противоречия в трафике куки: несколько платформ при одном UA.
Частота строки UA в модель не идёт.

## 7. Итог: наблюдение → признак

| Наблюдение | Группа признаков |
|---|---|
| файл не отсортирован, дубли, разный регистр `platform`, события после окна | очистка в `clean_events` |
| боты быстрее, особенно в хвостах распределения интервалов | `t_` тайминг |
| люди кликают равномерно, боты — блуждание с малым шагом и упорами в края; у ботов часто нет курсора | `p_` курсор |
| боты обходят больше городов и категорий | `c_` контент |
| боты глубже листают выдачу | `s_` поиск |
| боты реже листают фото, добавляют в избранное, логинятся | `e_` действия |
| куки ботов моложе, боты активнее ночью | `a_`, `h_` |
| тип клиента и противоречия UA ↔ платформа различаются; сырая строка — идентификатор | `u_` разобранный UA |

Что из этого реально работает в модели, показывает ablation в `train.py` (`results/experiments.csv`), а проверки
на утечки, сдвиг train/test и перенос на незнакомые типы ботов — в `validate.py` и `README.md`.